In [ ]:
RETRAIN = False

RUN_UMAP = False


# RD4AD - WideResNet50-2, x224

Reverse Distillation for Anomaly Detection using a frozen WideResNet50-2 encoder
and a trainable three-scale decoder. Trained exclusively on normal wafers.

**Default mode** (`RETRAIN = False`): loads the saved checkpoint and reuses
saved evaluation / UMAP artifacts when available.

**Retrain mode** (`RETRAIN = True`): runs `scripts/train_rd4ad.py` with the
checked-in `train_config.toml`, then re-evaluates.


## Imports and Paths


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image as IPyImage, display
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not find repo root containing src/wafer_defect')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'rd4ad' / 'wideresnet50' / 'x224' / 'main'
TRAIN_CONFIG = EXPERIMENT_DIR / 'train_config.toml'
ARTIFACT_DIR = EXPERIMENT_DIR / 'artifacts' / 'rd4ad_wrn50_x224'
RESULTS_DIR = ARTIFACT_DIR / 'results'
PLOTS_DIR = ARTIFACT_DIR / 'plots'
UMAP_DIR = RESULTS_DIR / 'umap'
CHECKPOINT_PATH = ARTIFACT_DIR / 'checkpoints' / 'best_model.pt'
METADATA_PATH = REPO_ROOT / 'data' / 'processed' / 'x224' / 'wm811k' / 'metadata_50k_5pct.csv'
RUNNER_SCRIPT = REPO_ROOT / 'scripts' / 'train_rd4ad.py'
NUM_WORKERS = 8
DEVICE_NAME = 'auto'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
UMAP_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo root : {REPO_ROOT}')
print(f'Artifact  : {ARTIFACT_DIR}')
print(f'Checkpoint: {CHECKPOINT_PATH} (exists={CHECKPOINT_PATH.exists()})')
print(f'UMAP dir   : {UMAP_DIR}')


## Optional Retraining

With `RETRAIN = False` the notebook skips directly to loading the saved checkpoint.


In [ ]:
if RETRAIN:
    cmd = [sys.executable, '-u', str(RUNNER_SCRIPT), '--config', str(TRAIN_CONFIG)]
    print('Launching training:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)
else:
    print('RETRAIN=False - skipping training, loading checkpoint.')
    if not CHECKPOINT_PATH.exists():
        print(f'WARNING: checkpoint not found at {CHECKPOINT_PATH}.\nSet RETRAIN=True to train from scratch.')


## Load Config and Metadata


In [ ]:
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
config = load_toml(str(TRAIN_CONFIG))
image_size = int(config['data'].get('image_size', 224))
batch_size = int(config['data'].get('batch_size', 64))
threshold_quantile = float(config['scoring'].get('threshold_quantile', 0.95))

def resolve_device(name: str) -> torch.device:
    if name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(name)

device = resolve_device(DEVICE_NAME)
metadata_ready = METADATA_PATH.exists()
evaluation_ready = False
if metadata_ready:
    metadata = pd.read_csv(METADATA_PATH)
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
else:
    metadata = pd.DataFrame()
    print(f'[WARNING] Benchmark metadata is missing: {METADATA_PATH}. RD4AD evaluation cells will be skipped unless saved artifacts already exist.')
print(f'Device: {device}  |  image_size: {image_size}  |  threshold_quantile: {threshold_quantile}')


## Load Model from Checkpoint


In [ ]:
from wafer_defect.models.rd4ad import build_rd4ad_from_config
model = None
if CHECKPOINT_PATH.exists():
    model = build_rd4ad_from_config(config).to(device)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_epoch = checkpoint.get('best_epoch', '?')
    best_val_loss = checkpoint.get('best_val_loss', float('nan'))
    print(f'Loaded checkpoint | best_epoch={best_epoch} | best_val_loss={best_val_loss:.6f}')
    model.eval()
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    total = sum((p.numel() for p in model.parameters()))
    print(f'Trainable parameters: {trainable:,} / {total:,}')
else:
    print(f'[WARNING] No checkpoint found at {CHECKPOINT_PATH}. Checkpoint-backed RD4AD cells will be skipped in this notebook run.')


## Training History


In [ ]:
history_path = RESULTS_DIR / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Total cosine loss')
    axes[0].legend()
    for col, label in [('train_l1', 'l1 (layer1)'), ('train_l2', 'l2 (layer2)'), ('train_l3', 'l3 (layer3)')]:
        axes[1].plot(history_df['epoch'], history_df[col], label=label)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Per-scale train losses')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(history_df.tail(5))
else:
    print('No history.json found - training has not been run yet.')


## Score Inference

Runs the model forward pass on the val and test splits to produce per-wafer anomaly scores.


In [ ]:
@torch.no_grad()
def infer_scores(split: str) -> tuple[np.ndarray, np.ndarray]:
    dataset = WaferMapDataset(str(METADATA_PATH), split=split, image_size=image_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    all_scores, all_labels = ([], [])
    for images, labels in loader:
        images = images.to(device)
        scores = model(images).cpu().numpy()
        all_scores.append(scores)
        all_labels.append(labels.numpy())
    return (np.concatenate(all_labels), np.concatenate(all_scores))

if model is None or not metadata_ready:
    print('[WARNING] RD4AD checkpoint-backed evaluation is unavailable because the checkpoint or metadata CSV is missing.')
else:
    val_labels, val_scores = infer_scores('val')
    test_labels, test_scores = infer_scores('test')
    evaluation_ready = True
    print(f'Val  : {len(val_labels):,} samples  |  anomaly rate={val_labels.mean():.3%}')
    print(f'Test : {len(test_labels):,} samples  |  anomaly rate={test_labels.mean():.3%}')


## Threshold and Metrics


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
    val_normal_scores = val_scores[val_labels == 0]
    threshold = float(np.quantile(val_normal_scores, threshold_quantile))
    print(f'Val-normal {threshold_quantile:.0%} threshold: {threshold:.6f}')
    val_metrics = summarize_threshold_metrics(val_labels, val_scores, threshold)
    test_metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
    metrics_df = pd.DataFrame([{'split': 'val', **{k: v for k, v in val_metrics.items() if k != 'confusion_matrix'}}, {'split': 'test', **{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}}])
    display(metrics_df.set_index('split'))
    pd.DataFrame({'label': val_labels, 'score': val_scores}).to_csv(RESULTS_DIR / 'val_scores.csv', index=False)
    pd.DataFrame({'label': test_labels, 'score': test_scores}).to_csv(RESULTS_DIR / 'test_scores.csv', index=False)
    summary = {'threshold': threshold, 'threshold_quantile': threshold_quantile, 'val': val_metrics, 'test': test_metrics}
    (RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"\nTest F1={test_metrics['f1']:.4f}  AUROC={test_metrics['auroc']:.4f}  AUPRC={test_metrics['auprc']:.4f}")


## Score Distribution Plot


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, labels, scores, split in [(axes[0], val_labels, val_scores, 'Val'), (axes[1], test_labels, test_scores, 'Test')]:
        ax.hist(scores[labels == 0], bins=80, alpha=0.6, label='Normal', density=True, color='steelblue')
        ax.hist(scores[labels == 1], bins=80, alpha=0.6, label='Defect', density=True, color='tomato')
        ax.axvline(threshold, color='black', linestyle='--', linewidth=1.2, label=f'Threshold ({threshold:.4f})')
        ax.set_title(f'{split} score distribution')
        ax.set_xlabel('Anomaly score')
        ax.set_ylabel('Density')
        ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()


## Threshold Sweep


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    sweep_df, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
    sweep_df.to_csv(RESULTS_DIR / 'threshold_sweep.csv', index=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sweep_df['threshold'], sweep_df['f1'], label='F1')
    ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', linestyle='--')
    ax.plot(sweep_df['threshold'], sweep_df['recall'], label='Recall', linestyle=':')
    ax.axvline(threshold, color='black', linestyle='--', linewidth=1.2, label=f'Val threshold ({threshold:.4f})')
    ax.axvline(best_sweep['threshold'], color='green', linestyle=':', linewidth=1.2, label=f"Oracle best F1 ({best_sweep['f1']:.4f})")
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Test threshold sweep')
    ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Val-normal threshold F1 : {test_metrics['f1']:.4f}")
    print(f"Oracle best-threshold F1: {best_sweep['f1']:.4f}")


## Confusion Matrix


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    cm = np.array(test_metrics['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred Normal', 'Pred Defect'])
    ax.set_yticklabels(['True Normal', 'True Defect'])
    ax.set_title('Test confusion matrix')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


## Per-Defect Recall


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    test_meta = metadata[metadata['split'] == 'test'].reset_index(drop=True)
    test_meta = test_meta.copy()
    test_meta['score'] = test_scores
    test_meta['predicted'] = (test_scores > threshold).astype(int)
    defect_label_col = 'failureType' if 'failureType' in test_meta.columns else None
    if defect_label_col:
        defect_rows = test_meta[test_meta['label'] == 1] if 'label' in test_meta.columns else test_meta[test_meta['split'] == 'test']
        true_defects = test_meta[test_labels == 1].copy()
        recall_by_defect = true_defects.groupby(defect_label_col).apply(lambda g: g['predicted'].mean()).rename('recall').reset_index().sort_values('recall', ascending=False)
        display(recall_by_defect)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(recall_by_defect[defect_label_col].astype(str), recall_by_defect['recall'], color='steelblue')
        ax.axhline(test_metrics['recall'], color='red', linestyle='--', label=f"Overall recall ({test_metrics['recall']:.3f})")
        ax.set_xlabel('Defect type')
        ax.set_ylabel('Recall')
        ax.set_title('Per-defect recall (test set)')
        ax.legend()
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'per_defect_recall.png', dpi=150, bbox_inches='tight')
        plt.show()
        recall_by_defect.to_csv(RESULTS_DIR / 'per_defect_recall.csv', index=False)
    else:
        print('No failureType column found in metadata - skipping per-defect breakdown.')


## UMAP Review

Loads saved embedding / UMAP artifacts by default.
to recompute them from the current checkpoint.


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    from wafer_defect.evaluation.umap_reference import export_reference_umap_bundle
    UMAP_POINTS_PATH = UMAP_DIR / 'umap_points.csv'
    UMAP_SUMMARY_PATH = UMAP_DIR / 'umap_summary.json'
    UMAP_SPLIT_PNG = UMAP_DIR / 'umap_by_split.png'
    UMAP_SCORE_PNG = UMAP_DIR / 'umap_by_score.png'

    @torch.no_grad()
    def infer_embeddings(split: str) -> tuple[np.ndarray, np.ndarray]:
        dataset = WaferMapDataset(str(METADATA_PATH), split=split, image_size=image_size)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
        all_embeddings, all_labels = ([], [])
        for images, labels in loader:
            images = images.to(device)
            _, _, f3 = model.encode(images)
            pooled = f3.mean(dim=(2, 3)).cpu().numpy().astype(np.float32)
            all_embeddings.append(pooled)
            all_labels.append(labels.numpy())
        return (np.concatenate(all_embeddings), np.concatenate(all_labels))
    umap_ready = UMAP_POINTS_PATH.exists() and UMAP_SUMMARY_PATH.exists() and UMAP_SPLIT_PNG.exists() and UMAP_SCORE_PNG.exists()
    if not RUN_UMAP and umap_ready:
        print('Loading saved UMAP artifacts ...')
    elif RUN_UMAP:
        print('Generating UMAP artifacts from checkpoint embeddings ...')
        import umap as umap_module
        train_embeddings, train_embedding_labels = infer_embeddings('train')
        val_embeddings, val_embedding_labels = infer_embeddings('val')
        test_embeddings, test_embedding_labels = infer_embeddings('test')
        np.save(UMAP_DIR / 'train_normal_embeddings.npy', train_embeddings)
        np.save(UMAP_DIR / 'train_normal_labels.npy', train_embedding_labels)
        np.save(UMAP_DIR / 'val_embeddings.npy', val_embeddings)
        np.save(UMAP_DIR / 'val_labels.npy', val_embedding_labels)
        np.save(UMAP_DIR / 'test_embeddings.npy', test_embeddings)
        np.save(UMAP_DIR / 'test_labels.npy', test_embedding_labels)
        export_reference_umap_bundle(output_dir=UMAP_DIR, umap_module=umap_module, train_normal_embeddings=train_embeddings, val_embeddings=val_embeddings, val_labels=val_embedding_labels, test_embeddings=test_embeddings, test_labels=test_embedding_labels, val_model_scores=val_scores.astype(np.float32), test_model_scores=test_scores.astype(np.float32), threshold_quantile=threshold_quantile, random_state=int(config['run'].get('seed', 42)), pca_components=50, n_neighbors=15, min_dist=0.1, knn_k=15, metric='euclidean', title_prefix='RD4AD WRN50 x224', points_filename='umap_points.csv', split_plot_filename='umap_by_split.png', score_plot_filename='umap_by_score.png', summary_filename='umap_summary.json', sweep_filename='umap_knn_threshold_sweep.csv')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    umap_points_df = pd.read_csv(UMAP_POINTS_PATH)
    umap_summary = json.loads(UMAP_SUMMARY_PATH.read_text(encoding='utf-8'))
    display(pd.DataFrame([{'umap_knn_threshold': umap_summary['umap_knn_threshold'], 'umap_knn_f1': umap_summary['umap_knn_metrics']['f1'], 'umap_knn_auroc': umap_summary['umap_knn_metrics']['auroc'], 'umap_knn_auprc': umap_summary['umap_knn_metrics']['auprc']}]))
    display(umap_points_df.head())
    if UMAP_SPLIT_PNG.exists():
        display(IPyImage(filename=str(UMAP_SPLIT_PNG)))
    if UMAP_SCORE_PNG.exists():
        display(IPyImage(filename=str(UMAP_SCORE_PNG)))


## Saved Outputs


In [ ]:
if not (evaluation_ready):
    print('[WARNING] RD4AD evaluation artifacts are unavailable because the checkpoint or metadata CSV is missing.')
else:
    saved = {'artifact_dir': str(ARTIFACT_DIR), 'results_dir': str(RESULTS_DIR), 'plots_dir': str(PLOTS_DIR), 'umap_dir': str(UMAP_DIR), 'checkpoint': str(CHECKPOINT_PATH), 'threshold': threshold, 'test_f1': test_metrics['f1'], 'test_auroc': test_metrics['auroc'], 'test_auprc': test_metrics['auprc']}
    saved
